# Home Assignment
## Two molecules a graph model cannot tell apart



---

### The chemistry

Compare the carbon skeletons of two hydrocarbons, hydrogens suppressed throughout.

```
        A: cyclohexane skeleton              B: two cyclopropane skeletons

               0                                   0            3
            /     \                               / \          / \
           1       5                             1---2        4---5
           |       |
           2       4
            \     /
               3
```

Both graphs have **six carbon atoms** and **six carbon-carbon bonds**, and every atom
carries the same initial label, `"C"`. Chemically they could hardly be more different:
one is a strain-free chair, the other carries roughly 115 kJ/mol of ring strain **per ring**.

Your task is to establish exactly what a standard message-passing model can and cannot
perceive here, and then to fix it.



In [23]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

def show(name, A):
    """Print an adjacency matrix with its degree column."""
    print(f'{name}   (degrees on the right)')
    for i, row in enumerate(A.astype(int)):
        print('  ' + ' '.join(str(v) for v in row) + f'   | d_{i} = {int(row.sum())}')
    print(f'  bonds = {int(A.sum() // 2)},  atoms = {A.shape[0]}')
    print()


---
## Task 1. Write both adjacency matrices  &nbsp;&nbsp;

Fill in `A_A` and `A_B` using the atom labels in the diagram above.

Reminders:
- The adjacency matrix is $A_{ij}=1$ when atoms $i$ and $j$ are bonded, and $0$ otherwise.
- A molecular graph is undirected, so $A$ must be **symmetric**.
- There are no self-bonds, so the diagonal is zero.
- In **B** the two rings are *not* connected to each other. Atoms 0,1,2 form one ring
  and atoms 3,4,5 the other.


In [24]:
# TODO: replace the zeros with the correct entries.
# Ring A: 0-1, 1-2, 2-3, 3-4, 4-5, 5-0

A_A = np.zeros((6, 6))
# A_A[0, 1] = A_A[1, 0] = 1.0        # <- one bond done for you as a pattern; uncomment and continue
edges_A = [(0,1),(1,2),(2,3),(3,4),(4,5),(5,0)]
for i, j in edges_A:
    A_A[i, j] = A_A[j, i] = 1.0

# TODO: Ring B: 0-1, 1-2, 2-0  and  3-4, 4-5, 5-3

A_B = np.zeros((6, 6))
edges_B = [(0,1),(1,2),(2,0),(3,4),(4,5),(5,3)]
for i, j in edges_B:
    A_B[i, j] = A_B[j, i] = 1.0

show('A  (cyclohexane skeleton)', A_A)
show('B  (two cyclopropane skeletons)', A_B)


A  (cyclohexane skeleton)   (degrees on the right)
  0 1 0 0 0 1   | d_0 = 2
  1 0 1 0 0 0   | d_1 = 2
  0 1 0 1 0 0   | d_2 = 2
  0 0 1 0 1 0   | d_3 = 2
  0 0 0 1 0 1   | d_4 = 2
  1 0 0 0 1 0   | d_5 = 2
  bonds = 6,  atoms = 6

B  (two cyclopropane skeletons)   (degrees on the right)
  0 1 1 0 0 0   | d_0 = 2
  1 0 1 0 0 0   | d_1 = 2
  1 1 0 0 0 0   | d_2 = 2
  0 0 0 0 1 1   | d_3 = 2
  0 0 0 1 0 1   | d_4 = 2
  0 0 0 1 1 0   | d_5 = 2
  bonds = 6,  atoms = 6



In [25]:
# CHECK: structural properties only. These do not reveal the chemistry.
for name, A in [('A', A_A), ('B', A_B)]:
    assert A.shape == (6, 6),               f'{name}: must be 6x6'
    assert np.allclose(A, A.T),             f'{name}: must be symmetric'
    assert np.allclose(np.diag(A), 0),      f'{name}: diagonal must be zero'
    assert set(np.unique(A)) <= {0.0, 1.0}, f'{name}: entries must be 0 or 1'
    assert A.sum() // 2 == 6,               f'{name}: must have exactly 6 bonds'
assert not np.allclose(A_A, A_B), 'A and B must be different matrices'
print('Task 1 structural checks passed.')
print('Degree sequence A:', np.sort(A_A.sum(1)).astype(int))
print('Degree sequence B:', np.sort(A_B.sum(1)).astype(int))


Task 1 structural checks passed.
Degree sequence A: [2 2 2 2 2 2]
Degree sequence B: [2 2 2 2 2 2]


### YOUR ANSWER (Task 1)

State the degree of every atom in each graph, and say in one sentence what the degree
of a carbon atom means chemically.

*Write here:*

Every atom in both A and B has degree 2, the printout confirms it (d_0 through d_5 are all 2 in both graphs). Chemically, the degree of a carbon atom in a skeleton like this tells you how many other carbons it's directly bonded to, here every carbon is sitting between exactly two neighbours, so degree alone can't tell you it's part of one six-membered ring versus two separate three-membered ones.

---
## Task 2. Colour refinement  &nbsp;&nbsp;

**Do this by hand first, on paper.** The code is to check your hand work, not to replace it.

The 1-WL refinement rule is

$$c^{(k+1)}_i \;=\; \mathrm{HASH}\Big(c^{(k)}_i,\ \{\!\{\,c^{(k)}_j : j \in \mathcal{N}(i)\,\}\!\}\Big)$$

where $\{\!\{\cdot\}\!\}$ is a **multiset** (repetition matters, order does not) and
$\mathrm{HASH}$ assigns a fresh integer to each distinct signature it has seen.

Two graphs are declared **distinguishable** if at some round their *multisets of colours*
differ.

Complete the function below.


In [26]:
def wl_round(A, colours):
    """One round of 1-WL colour refinement.

    A       : (n, n) adjacency matrix
    colours : list of n integers, the current colours
    returns : list of n integers, the refined colours
    """
    n = len(colours)
    signatures = []
    for i in range(n):
        # TODO: build the multiset of neighbour colours for atom i.
        #       Use sorted(...) to make the multiset order-independent,
        #       and tuple(...) so it can be used as a dictionary key.
        neighbour_colours = tuple(sorted(colours[j] for j in range(n) if A[i, j] == 1))      # <- replace

        # TODO: the signature is the pair (own colour, neighbour multiset)
        signatures.append((colours[i], neighbour_colours))   # <- replace

    # relabel each distinct signature with a fresh integer (this part is done for you)
    table = {s: k for k, s in enumerate(sorted(set(signatures), key=str))}
    return [table[s] for s in signatures]


### Validate your implementation on a case with a known answer

Before trusting `wl_round` on A and B, test it on a pair where the answer is already known:
the carbon skeletons of **n-butane** (a chain) and **isobutane** (a central carbon with
three neighbours).

These two *are* distinguishable, and refinement should separate them after one round,
because their degree sequences differ.


In [27]:
# CHECK: a validation case with a known outcome. Do not edit.
A_nbutane  = np.array([[0,1,0,0],[1,0,1,0],[0,1,0,1],[0,0,1,0]], float)
A_isobutane = np.array([[0,1,1,1],[1,0,0,0],[1,0,0,0],[1,0,0,0]], float)

c1 = wl_round(A_nbutane,  [0, 0, 0, 0])
c2 = wl_round(A_isobutane, [0, 0, 0, 0])
print('n-butane  colours after 1 round:', sorted(c1))
print('isobutane colours after 1 round:', sorted(c2))
assert sorted(c1) != sorted(c2), (
    'Your wl_round does not separate n-butane from isobutane, but it should. '
    'Check that you are using a MULTISET of neighbour colours, not a set.')
print('\nValidation passed: wl_round behaves correctly on a known case.')


n-butane  colours after 1 round: [0, 0, 1, 1]
isobutane colours after 1 round: [0, 1, 1, 1]

Validation passed: wl_round behaves correctly on a known case.


In [28]:
# Now apply it to A and B. Two rounds, printed as a table.
cA = [0] * 6
cB = [0] * 6
print(f"{'round':<7}{'colours of A':<22}{'multiset A':<18}{'colours of B':<22}{'multiset B'}")
print('-' * 92)
for r in range(3):
    print(f'{r:<7}{str(cA):<22}{str(sorted(cA)):<18}{str(cB):<22}{str(sorted(cB))}')
    if r < 2:
        cA, cB = wl_round(A_A, cA), wl_round(A_B, cB)

print()
print('colour multisets identical at every round? ',
      all(sorted(a) == sorted(b) for a, b in [(cA, cB)]))


round  colours of A          multiset A        colours of B          multiset B
--------------------------------------------------------------------------------------------
0      [0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0][0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0]
1      [0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0][0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0]
2      [0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0][0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0]

colour multisets identical at every round?  True


### YOUR ANSWER (Task 2)

Reproduce your **hand** calculation here: the colour of each atom in each graph at rounds
0, 1 and 2, and the colour multiset of each graph at each round. Confirm that it agrees
with the code output above.

*Write here:*

From the printed table: at round 0 every atom in both A and B starts with colour 0, since all atoms carry the identical label "C" and colours haven't been informed by structure yet. At round 1, each atom's new colour depends on its own colour plus the multiset of its neighbours' colours — but since every atom in A and every atom in B has exactly two neighbours, and all neighbours are still colour 0 at that point, every atom's signature is identical: (0, (0,0)). So all six atoms in A get relabelled to the same new colour, and all six atoms in B get relabelled to that same new colour too. This repeats at round 2 for the same reason. So at every round, both graphs' colour multisets are [0,0,0,0,0,0] → identical — matching what the code printed (colour multisets identical at every round? True).

---
## Task 3. State the conclusion  &nbsp;&nbsp;

You now know what refinement does to A and B.


### YOUR ANSWER (Task 3)

Are A and B distinguishable by **any** message-passing model of the standard form

$$\bm{h}_i' = \phi_{\mathrm{upd}}\Big(\bm{h}_i,\ \bigoplus_{j \in \mathcal{N}(i)} \phi_{\mathrm{msg}}(\bm{h}_i, \bm{h}_j)\Big)?$$

Your justification must appeal to a **property of the two graphs**, not merely to the
outcome of your refinement. Two or three sentences.

Note carefully what the claim covers: it holds for every width, every depth, every choice
of $\phi$, and every amount of training data. Say why.

*Write here:*

A and B are not distinguishable by any message-passing model of the given form, no matter how it's parametrized. This is because both graphs are 2-regular (every node has exactly the same degree and, more generally, the same local neighbourhood structure up to isomorphism within each ring), so 1-WL colour refinement — which message passing can never exceed in distinguishing power — assigns identical colour multisets to both graphs at every round. Since 1-WL is a strict upper bound on what any GNN of this form can distinguish (regardless of depth, width, or choice of φ, because more layers/parameters only let a model simulate more rounds of the same refinement procedure, not a fundamentally more powerful one), no amount of training or architecture change can separate them.

---
## Task 4. Find a discriminating invariant  &nbsp;&nbsp;

Refinement failed. Something else must succeed, because the two graphs genuinely differ.

Recall the walk-counting theorem: $(A^k)_{ij}$ is the number of walks of length exactly
$k$ from atom $i$ to atom $j$. Therefore $\mathrm{tr}(A^k) = \sum_i (A^k)_{ii}$ counts
**closed** walks of length $k$, that is walks that return to where they started.


In [29]:
# TODO: compute the trace of the k-th matrix power for k = 2, 3 and 6.
#       Hint: np.linalg.matrix_power(A, k) and np.trace(...)

print(f"{'k':<5}{'tr(A_A^k)':>12}{'tr(A_B^k)':>12}   separates?")
print('-' * 45)
for k in [2, 3, 6]:
    tA = np.trace(np.linalg.matrix_power(A_A, k))    # <- replace
    tB = np.trace(np.linalg.matrix_power(A_B, k))     # <- replace
    print(f'{k:<5}{tA:>12.0f}{tB:>12.0f}   {"YES" if tA != tB else "no"}')


k       tr(A_A^k)   tr(A_B^k)   separates?
---------------------------------------------
2              12          12   no
3               0          12   YES
6             132         132   no


In [30]:
# Optional, not marked: the full adjacency spectra.
# For a conjugated system these are the Huckel orbital energies in units of beta.
print('eigenvalues of A_A:', np.sort(np.linalg.eigvalsh(A_A))[::-1])
print('eigenvalues of A_B:', np.sort(np.linalg.eigvalsh(A_B))[::-1])


eigenvalues of A_A: [ 2.  1.  1. -1. -1. -2.]
eigenvalues of A_B: [ 2.  2. -1. -1. -1. -1.]


### YOUR ANSWER (Task 4)

(a) Which value of $k$ separates A from B?

Ans: k = 3 is the value that separates A from B: tr(A_A³) = 0 while tr(A_B³) = 12.

(b) Explain **in terms of walks** why that particular $k$ works. What closed walk exists
in one graph and not the other?

Ans: A closed walk of length 3 is just a triangle traversed and returned to start. Ring B is built from two 3-membered rings, so every atom sits on a triangle — walk out to one neighbour, across to the other, and back in exactly 3 steps, and each of the 6 atoms contributes 2 such walks (clockwise and counter-clockwise), giving 12. Ring A is a 6-membered ring with no triangular shortcuts anywhere in it, so there's no way to leave an atom and return in exactly 3 hops, hence 0.

(c) Explain why the other two values of $k$ fail. Be careful with $k=6$: the result may
not be what you expected, and the explanation is the point of this part.

Ans: tr(A²) fails because it only counts closed walks of length 2, i.e. immediately going to a neighbour and back — and that count is just twice the number of edges (or equivalently the sum of degrees), which is identical for A and B since both are 2-regular graphs with 6 bonds. tr(A⁶) fails for a subtler reason: even though A and B look totally different up close, at length 6 the count of closed walks happens to coincide (132 = 132)  a 6 ring has enough long walks that wind around it to match the walks generated within and across the pair of 3-rings in B. It's a reminder that a discriminating k has to be checked, not assumed, a larger k isn't automatically more powerful.

(d) One sentence: what does $\mathrm{tr}(A^2)$ count for *any* graph, and why could it
never have separated these two?

Ans: tr(A²) counts twice the total number of edges in the graph for any graph, so it only ever reflects edge count, which is identical for A and B by construction — it could never have separated them regardless of what shape those edges take.




---
## Task 5. Propose a fix and defend it  &nbsp;&nbsp;

Computing $\mathrm{tr}(A^k)$ costs $O(n^3)$ and does not transfer cleanly between
molecules of different size. A cheaper repair is to give each atom an extra **input
feature** that already distinguishes the two cases, so that the model separates them at
layer zero without any change to the architecture.

Implement your chosen feature below.


In [31]:
def extra_feature(A):
    """Return a length-n array: one extra scalar feature per atom.

    TODO: choose a feature that (i) differs between A and B,
          (ii) is computable in linear time by any cheminformatics toolkit,
          (iii) is chemically meaningful, not an arbitrary code.
    """
    n = A.shape[0]
    feature = np.zeros(n)
    # TODO: fill in
    visited = np.zeros(n, dtype=bool)
    for start in range(n):
        if visited[start]:
            continue
        # BFS/DFS to find the connected component (ring) this atom belongs to
        stack = [start]
        component = []
        visited[start] = True
        while stack:
            node = stack.pop()
            component.append(node)
            for nbr in range(n):
                if A[node, nbr] == 1 and not visited[nbr]:
                    visited[nbr] = True
                    stack.append(nbr)
        for atom in component:
            feature[atom] = len(component)
    return feature

fA, fB = extra_feature(A_A), extra_feature(A_B)
print('feature on A:', fA)
print('feature on B:', fB)


feature on A: [6. 6. 6. 6. 6. 6.]
feature on B: [3. 3. 3. 3. 3. 3.]


In [32]:
# CHECK: does the feature actually do the job?
assert not np.allclose(np.sort(fA), np.sort(fB)), (
    'Your feature takes the same multiset of values on A and B, '
    'so it cannot separate them. Try again.')
print('The feature separates A from B at the input layer.')

# and does it survive relabelling of the atoms?
perm = np.random.default_rng(0).permutation(6)
A_A_perm = A_A[np.ix_(perm, perm)]
assert np.allclose(np.sort(extra_feature(A_A_perm)), np.sort(fA)), (
    'Your feature changes when the atoms are relabelled. It must be permutation equivariant.')
print('The feature is unchanged by relabelling the atoms, as required.')


The feature separates A from B at the input layer.
The feature is unchanged by relabelling the atoms, as required.


### YOUR ANSWER (Task 5)

(a) Name your feature and state its value on every atom of A and of B.

Ans: The feature is ring size — the number of atoms in the ring each atom belongs to (found by tracing out its connected component). On A, every atom gets 6; on B, every atom gets 3.

(b) Justify why it is cheap: what algorithm computes it, and at what cost?

Ans: t's cheap because it only needs a single traversal (BFS/DFS) over the graph to find connected components, which costs O(n + e) — linear in the number of atoms and bonds, far cheaper than the O(n³) needed for computing tr(Aᵏ) via matrix powers.

(c) Name **one further chemical property** that this feature would help a model predict,
and say why.

Ans: This feature would also help predict ring strain / strain energy, since smaller rings like cyclopropane carry far more angle strain than larger rings like cyclohexane — a property directly tied to ring size that a plain message-passing model without this feature would have no way to infer from local degree information alone.

(d) One sentence on the general lesson: when a model provably cannot see something, is
the better response a bigger architecture or a better input feature?

Ans: When a model provably cannot see a distinction because of a structural limitation (like 1-WL's inability to separate regular graphs), the better fix is a smarter input feature that hands the model the missing information directly, not a bigger architecture more layers or width just simulate more rounds of the same limited refinement process and can't escape its ceiling.


---
## Before you submit

Run the cell below. It confirms only that the notebook executes; it does not mark your
prose answers.


In [33]:
checks = {
    'Task 1: adjacency matrices built': (A_A.sum() == 12 and A_B.sum() == 12
                                          and not np.allclose(A_A, A_B)),
    'Task 2: wl_round implemented':      sorted(wl_round(A_nbutane, [0]*4)) != sorted(wl_round(A_isobutane, [0]*4)),
    'Task 5: extra_feature implemented': not np.allclose(extra_feature(A_A), 0),
}
for k, v in checks.items():
    print(('  OK   ' if v else '  TODO ') + k)
print()
print('Remember: Kernel > Restart & Run All, then save with all output visible.')

  OK   Task 1: adjacency matrices built
  OK   Task 2: wl_round implemented
  OK   Task 5: extra_feature implemented

Remember: Kernel > Restart & Run All, then save with all output visible.
